# 04. Gunduz gallery и retrieval benchmark

Из detector test images формируются непересекающиеся support gallery и query. Оцениваются known, unseen species и unseen genus; название нового таксона всегда приходит из подписанного support-примера.

In [ ]:
from pathlib import Path
import os, subprocess, sys, yaml
import pandas as pd

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file()), None)
assert PROJECT_ROOT is not None
os.chdir(PROJECT_ROOT)
CONFIG = PROJECT_ROOT / 'configs/classifier_benchmark.yaml'
CHECKPOINT = PROJECT_ROOT / 'artifacts/classifier/dinov2/best.pt'
GALLERY_DIR = PROJECT_ROOT / 'artifacts/classifier/gunduz-gallery'
RUN_GALLERY_BUILD = False
RUN_RETRIEVAL_TEST = False
ENABLE_CLEARML = False

## Проверка протокола

In [ ]:
from core.config_loader import load_config
config = load_config(CONFIG)
table_path = PROJECT_ROOT / config['dataset']['table_path']
assert table_path.is_file(), table_path
benchmark = pd.read_csv(table_path)
display(benchmark.groupby('split').agg(crops=('id_crop', 'size'), genera=('genus', 'nunique'), species=('species', 'nunique')).reset_index())
if 'protocol_target' in benchmark.columns:
    display(pd.crosstab(benchmark['protocol_target'], benchmark['split']))
unit_column = 'image_id'
assert benchmark.groupby(unit_column)['split'].nunique().max() == 1, 'Один Gunduz image попал в gallery и query'
print(yaml.safe_dump(config, allow_unicode=True, sort_keys=False))

## Построение FAISS gallery

In [ ]:
gallery_command = [sys.executable, '-m', 'scripts.run_build_gallery', '--config', str(CONFIG), '--checkpoint', str(CHECKPOINT), '--output', str(GALLERY_DIR)]
if RUN_GALLERY_BUILD:
    assert CHECKPOINT.is_file(), CHECKPOINT
    subprocess.run(gallery_command, check=True)
else:
    print('Gallery build skipped:', ' '.join(map(str, gallery_command)))

## Retrieval test

In [ ]:
test_command = [sys.executable, '-m', 'scripts.run_test_classifier', '--config', str(CONFIG), '--checkpoint', str(CHECKPOINT)]
if ENABLE_CLEARML:
    test_command += ['--set', 'clearml.enabled=true']
if RUN_RETRIEVAL_TEST:
    assert CHECKPOINT.is_file(), CHECKPOINT
    assert (GALLERY_DIR / 'genus_index.npz').is_file()
    assert (GALLERY_DIR / 'species_index.npz').is_file()
    subprocess.run(test_command, check=True)
else:
    print('Retrieval test skipped:', ' '.join(map(str, test_command)))

## Отчёты

In [ ]:
output = PROJECT_ROOT / config['paths']['output_dir']
for path in sorted(output.rglob('*')) if output.exists() else []:
    if path.is_file(): print(path.relative_to(PROJECT_ROOT))